# 01 — Download raw Formula 1 data

## Purpose

Download, normalize, cache, and validate Formula 1 schedules, race results, qualifying results, and circuit reference data for 2004–2025.

## Inputs

- Jolpica F1 API (`https://api.jolpi.ca/ergast/f1`)
- Existing response caches and final CSVs, when available

## Outputs

- `data/raw/race_schedule.csv`
- `data/raw/race_results.csv`
- `data/raw/qualifying_results.csv`
- `data/raw/circuit_lookup.csv`
- `data/outputs/notebook_01_validation_report.json`

## Imports

In [1]:
from __future__ import annotations

import json
import pickle
import random
import time
from pathlib import Path
from typing import Any, Callable, Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from tqdm.auto import tqdm

## Configuration

In [2]:
START_YEAR = 2004
END_YEAR = 2025
FORCE_REFRESH = False

API_BASE_URL = "https://api.jolpi.ca/ergast/f1"
REQUEST_TIMEOUT_SECONDS = 30
REQUEST_SPACING_SECONDS = 0.20
MAX_RETRIES = 7
BACKOFF_BASE_SECONDS = 1.5
RETRIABLE_STATUS_CODES = {429, 500, 502, 503, 504}

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
RAW_DIR = PROJECT_ROOT / "data" / "raw"
CACHE_DIR = RAW_DIR / "api_cache"
OUTPUT_DIR = PROJECT_ROOT / "data" / "outputs"

FINAL_PATHS = {
    "race_schedule": RAW_DIR / "race_schedule.csv",
    "race_results": RAW_DIR / "race_results.csv",
    "qualifying_results": RAW_DIR / "qualifying_results.csv",
    "circuit_lookup": RAW_DIR / "circuit_lookup.csv",
}
VALIDATION_REPORT_PATH = OUTPUT_DIR / "notebook_01_validation_report.json"

for directory in (RAW_DIR, CACHE_DIR, OUTPUT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

session = requests.Session()
session.headers.update({"User-Agent": "f1-circuit-specialization/1.0 (portfolio analytics project)"})
session.mount("https://", HTTPAdapter(pool_connections=4, pool_maxsize=4))

## Helper functions

In [3]:
def nested_get(mapping: Mapping[str, Any], path: Sequence[str], default: Any = None) -> Any:
    """Return a nested mapping value without raising for missing keys."""
    value: Any = mapping
    for key in path:
        if not isinstance(value, Mapping) or key not in value:
            return default
        value = value[key]
    return value


def to_int(value: Any) -> Optional[int]:
    """Convert API integer-like values to nullable integers."""
    if value in (None, ""):
        return None
    try:
        return int(value)
    except (TypeError, ValueError):
        return None


def cache_path(category: str, filename: str) -> Path:
    """Build and create a category-specific response-cache path."""
    directory = CACHE_DIR / category
    directory.mkdir(parents=True, exist_ok=True)
    return directory / filename


def request_json(url: str, destination: Path, force_refresh: bool = False) -> Dict[str, Any]:
    """Load cached JSON or request it with retries, backoff, and immediate persistence."""
    if destination.exists() and not force_refresh:
        with destination.open("rb") as handle:
            return pickle.load(handle)

    for attempt in range(MAX_RETRIES):
        try:
            response = session.get(url, timeout=REQUEST_TIMEOUT_SECONDS)
            if response.status_code in RETRIABLE_STATUS_CODES:
                retry_after = response.headers.get("Retry-After")
                delay = float(retry_after) if retry_after else BACKOFF_BASE_SECONDS * (2 ** attempt)
                delay += random.uniform(0.0, 0.5)
                if attempt == MAX_RETRIES - 1:
                    response.raise_for_status()
                time.sleep(delay)
                continue
            response.raise_for_status()
            payload = response.json()
            temporary_path = destination.with_suffix(destination.suffix + ".tmp")
            with temporary_path.open("wb") as handle:
                pickle.dump(payload, handle, protocol=pickle.HIGHEST_PROTOCOL)
            temporary_path.replace(destination)
            time.sleep(REQUEST_SPACING_SECONDS)
            return payload
        except (requests.RequestException, ValueError) as exc:
            if attempt == MAX_RETRIES - 1:
                raise RuntimeError(f"Request failed after {MAX_RETRIES} attempts: {url}") from exc
            time.sleep(BACKOFF_BASE_SECONDS * (2 ** attempt) + random.uniform(0.0, 0.5))
    raise RuntimeError(f"Unreachable retry state for {url}")


def extract_races(payload: Mapping[str, Any]) -> List[Dict[str, Any]]:
    """Extract the standard Jolpica race list from a response."""
    races = nested_get(payload, ("MRData", "RaceTable", "Races"), [])
    if not isinstance(races, list):
        raise ValueError("Unexpected API response: RaceTable.Races is not a list")
    return races


def nullable_int_columns(frame: pd.DataFrame, columns: Iterable[str]) -> pd.DataFrame:
    """Apply pandas nullable integer dtype to selected columns."""
    for column in columns:
        if column in frame.columns:
            frame[column] = pd.to_numeric(frame[column], errors="coerce").astype("Int64")
    return frame


def save_csv(frame: pd.DataFrame, path: Path) -> None:
    """Write a CSV atomically to avoid leaving a partial final file."""
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temporary_path, index=False)
    temporary_path.replace(path)


def report_check(name: str, passed: bool, details: Any) -> Dict[str, Any]:
    """Create a JSON-serializable validation-check record."""
    return {"check": name, "passed": bool(passed), "details": details}


def cache_count(category: str) -> int:
    """Count completed pickle caches belonging to the configured season scope."""
    cache_files = (CACHE_DIR / category).glob("*.pkl")
    season_tokens = {f"_{season}" for season in range(START_YEAR, END_YEAR + 1)}
    return sum(
        any(token in cache_file.stem for token in season_tokens)
        for cache_file in cache_files
    )


def filter_season_scope(frame: pd.DataFrame) -> pd.DataFrame:
    """Return only rows inside the configured inclusive season range."""
    if "season" not in frame.columns:
        return frame.copy()
    seasons = pd.to_numeric(frame["season"], errors="coerce")
    return frame.loc[seasons.between(START_YEAR, END_YEAR)].reset_index(drop=True)

## Processing

In [4]:
all_final_files_exist = all(path.exists() for path in FINAL_PATHS.values())
use_final_csv_cache = all_final_files_exist and not FORCE_REFRESH

if use_final_csv_cache:
    print("Final CSV cache found; skipping all API downloads.")
    race_schedule = filter_season_scope(
        pd.read_csv(FINAL_PATHS["race_schedule"], dtype={"season": "Int64", "round": "Int64"})
    )
    race_results = filter_season_scope(pd.read_csv(FINAL_PATHS["race_results"]))
    qualifying_results = filter_season_scope(pd.read_csv(FINAL_PATHS["qualifying_results"]))
    circuit_lookup = pd.read_csv(FINAL_PATHS["circuit_lookup"])
    circuit_lookup = circuit_lookup[
        circuit_lookup["circuit_id"].isin(race_schedule["circuit_id"])
    ].reset_index(drop=True)
else:
    schedule_rows: List[Dict[str, Any]] = []
    for season in tqdm(range(START_YEAR, END_YEAR + 1), desc="Schedules"):
        url = f"{API_BASE_URL}/{season}.json?limit=100"
        payload = request_json(url, cache_path("schedule", f"schedule_{season}.pkl"), FORCE_REFRESH)
        for race in extract_races(payload):
            circuit = race.get("Circuit", {})
            location = circuit.get("Location", {})
            schedule_rows.append({
                "season": to_int(race.get("season")), "round": to_int(race.get("round")),
                "race_name": race.get("raceName"), "date": race.get("date"), "time": race.get("time"),
                "url": race.get("url"), "circuit_id": circuit.get("circuitId"),
                "circuit_name": circuit.get("circuitName"), "circuit_url": circuit.get("url"),
                "locality": location.get("locality"), "country": location.get("country"),
                "latitude": location.get("lat"), "longitude": location.get("long"),
            })
    race_schedule = nullable_int_columns(pd.DataFrame(schedule_rows), ["season", "round"])
    race_schedule = race_schedule.sort_values(["season", "round"]).reset_index(drop=True)

    result_rows: List[Dict[str, Any]] = []
    qualifying_rows: List[Dict[str, Any]] = []
    race_keys = list(race_schedule[["season", "round"]].itertuples(index=False, name=None))
    for season, round_number in tqdm(race_keys, desc="Race results"):
        url = f"{API_BASE_URL}/{season}/{round_number}/results.json?limit=100"
        payload = request_json(url, cache_path("race_results", f"race_results_{season}_{int(round_number):02d}.pkl"), FORCE_REFRESH)
        for race in extract_races(payload):
            for result in race.get("Results", []):
                driver, constructor = result.get("Driver", {}), result.get("Constructor", {})
                fastest = result.get("FastestLap", {})
                result_rows.append({
                    "season": to_int(race.get("season")), "round": to_int(race.get("round")),
                    "race_name": race.get("raceName"), "circuit_id": nested_get(race, ("Circuit", "circuitId")),
                    "driver_id": driver.get("driverId"), "driver_code": driver.get("code"),
                    "permanent_number": driver.get("permanentNumber"), "given_name": driver.get("givenName"),
                    "family_name": driver.get("familyName"), "date_of_birth": driver.get("dateOfBirth"),
                    "driver_nationality": driver.get("nationality"), "constructor_id": constructor.get("constructorId"),
                    "constructor_name": constructor.get("name"), "constructor_nationality": constructor.get("nationality"),
                    "car_number": result.get("number"), "position": to_int(result.get("position")),
                    "position_text": result.get("positionText"), "grid": to_int(result.get("grid")),
                    "points": result.get("points"), "laps": to_int(result.get("laps")),
                    "status": result.get("status"), "race_time_millis": nested_get(result, ("Time", "millis")),
                    "race_time": nested_get(result, ("Time", "time")), "fastest_lap_rank": to_int(fastest.get("rank")),
                    "fastest_lap_number": to_int(fastest.get("lap")), "fastest_lap_time": nested_get(fastest, ("Time", "time")),
                    "fastest_lap_avg_speed": nested_get(fastest, ("AverageSpeed", "speed")),
                    "fastest_lap_speed_units": nested_get(fastest, ("AverageSpeed", "units")),
                })

    for season, round_number in tqdm(race_keys, desc="Qualifying results"):
        url = f"{API_BASE_URL}/{season}/{round_number}/qualifying.json?limit=100"
        payload = request_json(url, cache_path("qualifying_results", f"qualifying_results_{season}_{int(round_number):02d}.pkl"), FORCE_REFRESH)
        for race in extract_races(payload):
            for result in race.get("QualifyingResults", []):
                driver, constructor = result.get("Driver", {}), result.get("Constructor", {})
                qualifying_rows.append({
                    "season": to_int(race.get("season")), "round": to_int(race.get("round")),
                    "race_name": race.get("raceName"), "circuit_id": nested_get(race, ("Circuit", "circuitId")),
                    "driver_id": driver.get("driverId"), "driver_code": driver.get("code"),
                    "given_name": driver.get("givenName"), "family_name": driver.get("familyName"),
                    "constructor_id": constructor.get("constructorId"), "constructor_name": constructor.get("name"),
                    "car_number": result.get("number"), "qualifying_position": to_int(result.get("position")),
                    "q1": result.get("Q1"), "q2": result.get("Q2"), "q3": result.get("Q3"),
                })

    race_results = nullable_int_columns(pd.DataFrame(result_rows), ["season", "round", "position", "grid", "laps", "fastest_lap_rank", "fastest_lap_number"])
    qualifying_results = nullable_int_columns(pd.DataFrame(qualifying_rows), ["season", "round", "qualifying_position"])
    race_results = race_results.sort_values(["season", "round", "position", "driver_id"]).reset_index(drop=True)
    qualifying_results = qualifying_results.sort_values(["season", "round", "qualifying_position", "driver_id"]).reset_index(drop=True)
    circuit_lookup = (race_schedule[["circuit_id", "circuit_name", "circuit_url", "locality", "country", "latitude", "longitude"]]
                      .drop_duplicates("circuit_id").sort_values("circuit_id").reset_index(drop=True))

Schedules:   0%|          | 0/22 [00:00<?, ?it/s]

Race results:   0%|          | 0/436 [00:00<?, ?it/s]

Qualifying results:   0%|          | 0/436 [00:00<?, ?it/s]

## Diagnostics

In [5]:
diagnostics = pd.DataFrame({
    "metric": ["seasons", "scheduled_races", "race_result_rows", "qualifying_rows", "drivers", "constructors", "circuits"],
    "value": [race_schedule["season"].nunique(), len(race_schedule), len(race_results), len(qualifying_results),
              race_results["driver_id"].nunique(), race_results["constructor_id"].nunique(), race_schedule["circuit_id"].nunique()],
})
display(diagnostics)
display(race_schedule.groupby("season").size().rename("scheduled_races").to_frame().T)
print({category: cache_count(category) for category in ("schedule", "race_results", "qualifying_results")})

,metric,value
0,seasons,22
1,scheduled_races,436
2,race_result_rows,9125
3,qualifying_rows,9105
4,drivers,111
5,constructors,35
6,circuits,38


season,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
scheduled_races,18,19,18,17,18,17,19,19,20,19,...,21,20,21,21,17,22,22,22,24,24


{'schedule': 22, 'race_results': 436, 'qualifying_results': 436}


## Save outputs

In [6]:
for name, frame in {
    "race_schedule": race_schedule,
    "race_results": race_results,
    "qualifying_results": qualifying_results,
    "circuit_lookup": circuit_lookup,
}.items():
    save_csv(frame, FINAL_PATHS[name])
print("Saved:", *(str(path.relative_to(PROJECT_ROOT)) for path in FINAL_PATHS.values()), sep="\n- ")

Saved:
- data/raw/race_schedule.csv
- data/raw/race_results.csv
- data/raw/qualifying_results.csv
- data/raw/circuit_lookup.csv


## Validation

In [7]:
EXPECTED_COLUMNS = {
    "race_schedule": {"season", "round", "race_name", "circuit_id", "date"},
    "race_results": {"season", "round", "driver_id", "constructor_id", "position", "grid", "status", "laps"},
    "qualifying_results": {"season", "round", "driver_id", "qualifying_position", "q1", "q2", "q3"},
    "circuit_lookup": {"circuit_id", "circuit_name", "locality", "country"},
}
frames = {"race_schedule": race_schedule, "race_results": race_results,
          "qualifying_results": qualifying_results, "circuit_lookup": circuit_lookup}
schedule_keys = set(map(tuple, race_schedule[["season", "round"]].astype(int).to_numpy()))
result_keys = set(map(tuple, race_results[["season", "round"]].astype(int).drop_duplicates().to_numpy()))
qualifying_keys = set(map(tuple, qualifying_results[["season", "round"]].astype(int).drop_duplicates().to_numpy()))
missing_result_races = sorted(schedule_keys - result_keys)
missing_qualifying_races = sorted(schedule_keys - qualifying_keys)

checks = [
    report_check("season coverage", set(race_schedule["season"].dropna().astype(int)) == set(range(START_YEAR, END_YEAR + 1)),
                 {"min": int(race_schedule["season"].min()), "max": int(race_schedule["season"].max())}),
    report_check("no duplicate schedule rows", not race_schedule.duplicated(["season", "round"]).any(),
                 int(race_schedule.duplicated(["season", "round"]).sum())),
    report_check("no duplicate driver-race results", not race_results.duplicated(["season", "round", "driver_id"]).any(),
                 int(race_results.duplicated(["season", "round", "driver_id"]).sum())),
    report_check("no duplicate driver-race qualifying rows", not qualifying_results.duplicated(["season", "round", "driver_id"]).any(),
                 int(qualifying_results.duplicated(["season", "round", "driver_id"]).sum())),
    report_check("all scheduled races have results", len(missing_result_races) == 0, missing_result_races),
    report_check("all scheduled races have qualifying", len(missing_qualifying_races) == 0, missing_qualifying_races),
    report_check("expected columns", all(columns <= set(frames[name].columns) for name, columns in EXPECTED_COLUMNS.items()),
                 {name: sorted(columns - set(frames[name].columns)) for name, columns in EXPECTED_COLUMNS.items()}),
    report_check("nonempty entities", race_results["driver_id"].nunique() > 0 and race_results["constructor_id"].nunique() > 0 and circuit_lookup["circuit_id"].nunique() > 0,
                 {"drivers": int(race_results["driver_id"].nunique()), "constructors": int(race_results["constructor_id"].nunique()), "circuits": int(circuit_lookup["circuit_id"].nunique())}),
]

if not use_final_csv_cache:
    checks.append(report_check("schedule cache coverage", cache_count("schedule") == END_YEAR - START_YEAR + 1,
                               {"actual": cache_count("schedule"), "expected": END_YEAR - START_YEAR + 1}))
    checks.append(report_check("race-result cache coverage", cache_count("race_results") == len(race_schedule),
                               {"actual": cache_count("race_results"), "expected": len(race_schedule)}))
    checks.append(report_check("qualifying cache coverage", cache_count("qualifying_results") == len(race_schedule),
                               {"actual": cache_count("qualifying_results"), "expected": len(race_schedule)}))

validation_report = {
    "notebook": "01_download_raw_data", "scope": {"start_year": START_YEAR, "end_year": END_YEAR},
    "status": "PASS" if all(check["passed"] for check in checks) else "FAIL", "checks": checks,
}
display(pd.DataFrame(checks))
if validation_report["status"] != "PASS":
    failed = [check for check in checks if not check["passed"]]
    raise AssertionError(f"Validation failed: {failed}")
print("PASS")

,check,passed,details
0,season coverage,True,"{'min': 2004, 'max': 2025}"
1,no duplicate schedule rows,True,0
2,no duplicate driver-race results,True,0
3,no duplicate driver-race qualifying rows,True,0
4,all scheduled races have results,True,[]
5,all scheduled races have qualifying,True,[]
6,expected columns,True,"{'race_schedule': [], 'race_results': [], 'qua..."
7,nonempty entities,True,"{'drivers': 111, 'constructors': 35, 'circuits..."
8,schedule cache coverage,True,"{'actual': 22, 'expected': 22}"
9,race-result cache coverage,True,"{'actual': 436, 'expected': 436}"


PASS


## Supercheck

In [8]:
def normalized_csv(frame: pd.DataFrame) -> pd.DataFrame:
    """Normalize a frame through CSV serialization for an integrity comparison."""
    return frame.reset_index(drop=True).astype("string").fillna("").astype(str)


superchecks = []
for name, path in FINAL_PATHS.items():
    exists = path.exists()
    saved = pd.read_csv(path, dtype=str, keep_default_na=False) if exists else pd.DataFrame()
    equal = exists and normalized_csv(frames[name]).equals(normalized_csv(saved))
    superchecks.append(report_check(f"{name} file exists", exists, str(path)))
    superchecks.append(report_check(f"{name} saved data equals memory", equal, {"rows": len(saved)}))

validation_report["superchecks"] = superchecks
validation_report["status"] = "PASS" if all(check["passed"] for check in checks + superchecks) else "FAIL"
with VALIDATION_REPORT_PATH.open("w", encoding="utf-8") as handle:
    json.dump(validation_report, handle, indent=2, default=str)

report_roundtrip = json.loads(VALIDATION_REPORT_PATH.read_text(encoding="utf-8"))
assert VALIDATION_REPORT_PATH.exists(), "Validation report was not saved"
assert report_roundtrip == validation_report, "Saved validation report differs from memory"
display(pd.DataFrame(superchecks))
if validation_report["status"] != "PASS":
    raise AssertionError("Supercheck failed")
print("PASS")

,check,passed,details
0,race_schedule file exists,True,
1,race_schedule saved data equals memory,True,{'rows': 436}
2,race_results file exists,True,
3,race_results saved data equals memory,True,{'rows': 9125}
4,qualifying_results file exists,True,
5,qualifying_results saved data equals memory,True,{'rows': 9105}
6,circuit_lookup file exists,True,
7,circuit_lookup saved data equals memory,True,{'rows': 38}


PASS
